# Customer Data EDA & Feature Engineering

## Setup

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

DB_PATH = '../data/churn_analytics.db'
ANALYSIS_DATE = '2026-08-01'

## Load Raw Data

In [ ]:
conn = sqlite3.connect(DB_PATH)

customers = pd.read_sql('SELECT * FROM customers', conn)
subscriptions = pd.read_sql('SELECT * FROM subscriptions', conn)
transactions = pd.read_sql('SELECT * FROM transactions', conn)

customers.head()

## Data Cleaning

In [ ]:
transactions = transactions.drop_duplicates(
    subset=['customer_id', 'transaction_date', 'amount', 'payment_method']
)
transactions = transactions[transactions['amount'] > 0]

valid_ids = set(customers['customer_id'])
transactions = transactions[transactions['customer_id'].isin(valid_ids)]
subscriptions = subscriptions[subscriptions['customer_id'].isin(valid_ids)]

for col in ['region', 'acquisition_channel']:
    customers[col] = customers[col].str.strip()

subscriptions['plan_type'] = subscriptions['plan_type'].str.strip()
subscriptions['status'] = subscriptions['status'].str.strip()

## Missing Values Audit

In [ ]:
print('Customers nulls:\n', customers.isna().sum())
print('\nSubscriptions nulls:\n', subscriptions.isna().sum())
print('\nTransactions nulls:\n', transactions.isna().sum())

## Feature Engineering

In [ ]:
tx_agg = transactions.groupby('customer_id').agg(
    frequency=('transaction_id', 'count'),
    total_revenue=('amount', 'sum'),
    last_transaction_date=('transaction_date', 'max'),
).reset_index()

df = customers.merge(subscriptions, on='customer_id', how='inner')
df = df.merge(tx_agg, on='customer_id', how='left')
df['frequency'] = df['frequency'].fillna(0)
df['total_revenue'] = df['total_revenue'].fillna(0)

analysis_ts = pd.Timestamp(ANALYSIS_DATE)
df['recency_days'] = (analysis_ts - pd.to_datetime(df['last_transaction_date'])).dt.days
df['tenure_days'] = (pd.to_datetime(df['end_date'].fillna(ANALYSIS_DATE)) - pd.to_datetime(df['start_date'])).dt.days
df['is_churned'] = (df['status'] == 'Cancelled').astype(int)

df['avg_region_revenue'] = df.groupby('region')['total_revenue'].transform('mean')
df['region_spend_rank'] = df.groupby('region')['total_revenue'].rank(ascending=False, method='first')

df.head()

## Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df['recency_days'], bins=30, ax=axes[0]).set_title('Recency (days)')
sns.histplot(df['frequency'], bins=30, ax=axes[1]).set_title('Frequency')
sns.histplot(df['total_revenue'], bins=30, ax=axes[2]).set_title('Monetary Value')
plt.tight_layout()
plt.show()

In [ ]:
churn_by_plan = df.groupby('plan_type')['is_churned'].mean().sort_values(ascending=False)
churn_by_plan.plot(kind='bar', title='Churn Rate by Plan Type')
plt.ylabel('Churn Rate')
plt.show()

In [ ]:
churn_by_region = df.groupby('region')['is_churned'].mean().sort_values(ascending=False)
churn_by_region.plot(kind='bar', title='Churn Rate by Region')
plt.ylabel('Churn Rate')
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
corr_cols = ['recency_days', 'frequency', 'total_revenue', 'tenure_days', 'monthly_price', 'is_churned']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

## Export Data

In [ ]:
df.to_csv('../data/cleaned_customer_data.csv', index=False)
conn.close()